# NEOFC - Prepare phenotypic data for HCP-Early Psychosis dataset

In [2]:
from pathlib import Path 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
na = slice(None)

# working directory
wd = Path.cwd() 
print(wd)

# general vars
from utils import (REF_GROUPS, REF_GROUPS_COLORS, REF_GROUPNAMES_PET, REF_NAMES_PET, REF_GROUPS_PET,
                   PARC_DEFAULT, PARCS_ALL, PARCS_CX,
                   MEASURES_ALL, MEASURES_NICE)


/Users/llotter/projects/mapfc


## Prep qc data

In [3]:

qc_hcp_ep = pd.read_table(wd / "data_source" / "timeseries" / "hcp_ep" / "linc_qc.tsv")
qc_hcp_ep = qc_hcp_ep[["sub", "dir", "mean_fd"]]

# handle index
qc_hcp_ep["sub"] = qc_hcp_ep["sub"].astype(str)
qc_hcp_ep = qc_hcp_ep.set_index(["sub", "dir"])

# qc threshold
qc_hcp_ep["qc_pass"] = qc_hcp_ep["mean_fd"] <= 0.3

# average per direction
qc_hcp_ep = qc_hcp_ep.groupby(level="sub").mean()
qc_hcp_ep.qc_pass = qc_hcp_ep.qc_pass == 1.0

# add "sub-" to index
qc_hcp_ep.index = "sub-" + qc_hcp_ep.index.astype(str)

# save
qc_hcp_ep.to_csv(wd / "data_deriv" / "pheno" / "hcp_ep" / "hcpep_qc.csv")

qc_hcp_ep.qc_pass.value_counts()

qc_pass
True     153
False      8
Name: count, dtype: int64

## Prep phenotype data

In [6]:
# pheno dir
pheno_dir = wd / "data_source" / "pheno" / "hcp_ep"

# load individual tables and rename columns
metadata_hcp_ep = (
    pd.read_table(pheno_dir / "ndar_subject01.txt", skiprows=[1])
    .merge(
        pd.read_table(pheno_dir / "ses01.txt", skiprows=[1])[["src_subject_id", "sestot"]],
        on="src_subject_id",
        how="left",
        suffixes=(None, "_ses")
    )
    .merge(
        pd.read_table(pheno_dir / "scid_v01.txt", skiprows=[1]),
        on="src_subject_id",
        how="left",
        suffixes=(None, "_scid")
    )
    .merge(
        pd.read_table(pheno_dir / "panss01.txt", skiprows=[1]),
        on="src_subject_id",
        how="left",
        suffixes=(None, "_panss")
    )
    .merge(
        pd.read_table(pheno_dir / "mhx01.txt", skiprows=[1]),
        on="src_subject_id",
        how="left",
        suffixes=(None, "_mhx")
    )
    .merge(
        pd.read_table(pheno_dir / "cogcomp01.txt", skiprows=[1], na_values=["999"]),
        on="src_subject_id",
        how="left",
        suffixes=(None, "_cogcomp")
    )
    .merge(
        pd.read_table(pheno_dir / "lswmt01.txt", skiprows=[1]),
        on="src_subject_id",
        how="left",
        suffixes=(None, "_lswmt")
    )
    .merge(
        pd.read_table(pheno_dir / "pcps01.txt", skiprows=[1]),
        on="src_subject_id",
        how="left",
        suffixes=(None, "_pcps")
    )
    .merge(
        pd.read_table(pheno_dir / "fmhx01.txt", skiprows=[1]),
        on="src_subject_id",
        how="left",
        suffixes=(None, "_fmhx")
    )
)
metadata_hcp_ep = metadata_hcp_ep[
    ["src_subject_id", "sex", "interview_age", "race", "site", "phenotype_description"] \
    + ["sestot"] \
    + [f"pos_p{i}" for i in range(1, 8)] \
    + [f"neg_n{i}" for i in range(1, 8)] \
    + [f"gps_g{i}" for i in range(1, 17)] \
    + ["apd_exp_months", "apd_chlor_equiv"] \
    + ["nih_fluidcogcomp_unadjusted", "nih_fluidcogcomp_ageadjusted", "nih_crycogcomp_unadjusted", "nih_crycogcomp_ageadjusted", "nih_totalcogcomp_unadjusted", "nih_totalcogcomp_ageadjusted"] \
    + ["tbx_ls"] \
    + ["nih_patterncomp_unadjusted", "nih_patterncomp_ageadjusted"] \
    + ["q042_sz_life", "q043_sz_mon", "q045_szp_life", "q046_szp_mon", "q048_sza_life", "q049_sza_mon", 
       "q051_dd_life", "q052_dd_mon", "q053_bfpd_life", "q054_bfpd_mon",
       "q055_pdgmc_life", "q057_pdgmc_mon", "q060_sipd_life", "q062_sipd_mon", "scid_p46", "scid_p47"] \
    + ["fahxp1n", "fahxp2n"]
]
metadata_hcp_ep = metadata_hcp_ep.rename(
    columns={
        "src_subject_id": "sub", 
        "interview_age": "age",
        "phenotype_description": "dx_aff"
    }
)

# handle subject
metadata_hcp_ep["sub"] = "sub-" + metadata_hcp_ep["sub"].astype(str)
metadata_hcp_ep = metadata_hcp_ep.set_index("sub")

# handle age
metadata_hcp_ep["age"] = (metadata_hcp_ep["age"] / 12).round(2)

# handle sex
metadata_hcp_ep["sex"] = pd.Categorical(metadata_hcp_ep["sex"])
metadata_hcp_ep["sex_num"] = metadata_hcp_ep["sex"].cat.codes

# handle site
metadata_hcp_ep["site"] = pd.Categorical(metadata_hcp_ep["site"])
metadata_hcp_ep["site_num"] = metadata_hcp_ep["site"].cat.codes

# handle diagnosis: CTRL, aPS, PS
metadata_hcp_ep["dx_sub"] = pd.Categorical(
    metadata_hcp_ep["dx_aff"]
    .replace({
            "In good health": "CTRL",
            "Affective psychosis": "SCA",
            "Non-affective psychosis": "SCZ"
    }),
    categories=["CTRL", "SCZ", "SCA"],
    ordered=True
)

# simplified diagnoses
metadata_hcp_ep["dx"] = metadata_hcp_ep["dx_sub"].map({
    "CTRL": "CTRL",
    "SCZ": "SCZ",
    "SCF": "SCZ",
    "SCA": "SCZ"
})
display(metadata_hcp_ep["dx"].value_counts(dropna=False))

# remove subjects with current medication or history of medication
metadata_hcp_ep = metadata_hcp_ep.query("(dx == 'SCZ') | ((dx == 'CTRL') & (apd_chlor_equiv == 0) & (apd_exp_months == 0))")
display(metadata_hcp_ep["dx"].value_counts(dropna=False))

# currently medicated
metadata_hcp_ep["apd_current"] = metadata_hcp_ep["apd_chlor_equiv"] > 0

# ever medicated
metadata_hcp_ep["apd_lifetime"] = (metadata_hcp_ep["apd_chlor_equiv"] > 0) | (metadata_hcp_ep["apd_exp_months"] > 0)

# diagnosis * current medicated
metadata_hcp_ep["dx_apd_current"] = (metadata_hcp_ep["dx"] + "_" + metadata_hcp_ep["apd_current"].astype(str)).map({
    "SCZ_True": "SCZ_apd",
    "SCZ_False": "SCZ_noapd",
    "CTRL_False": "CTRL",
})
display(metadata_hcp_ep["dx_apd_current"].value_counts(dropna=False))

# diagnosis * ever medicated
metadata_hcp_ep["dx_apd_lifetime"] = (metadata_hcp_ep["dx"] + "_" + metadata_hcp_ep["apd_lifetime"].astype(str)).map({
    "SCZ_True": "SCZ_apd",
    "SCZ_False": "SCZ_noapd",
    "CTRL_False": "CTRL",
})
display(metadata_hcp_ep["dx_apd_lifetime"].value_counts(dropna=False))

# handle PANSS
metadata_hcp_ep["panss_pos"] = metadata_hcp_ep[metadata_hcp_ep.filter(like="pos_").columns].sum(axis=1)
metadata_hcp_ep["panss_neg"] = metadata_hcp_ep[metadata_hcp_ep.filter(like="neg_").columns].sum(axis=1)
metadata_hcp_ep["panss_gps"] = metadata_hcp_ep[metadata_hcp_ep.filter(like="gps_").columns].sum(axis=1)
metadata_hcp_ep["panss_total"] = metadata_hcp_ep[["panss_pos", "panss_neg", "panss_gps"]].sum(axis=1)
metadata_hcp_ep = metadata_hcp_ep.drop(columns=metadata_hcp_ep.filter(regex="pos_|neg_|gps_").columns)
metadata_hcp_ep.loc[metadata_hcp_ep["panss_total"] == 0, ["panss_pos", "panss_neg", "panss_gps", "panss_total"]] = np.nan

# save
metadata_hcp_ep.to_csv(wd / "data_deriv" / "pheno" / "hcp_ep" / "hcpep_pheno.csv")

# check
#display(metadata_hcp_ep.head())
display(metadata_hcp_ep.groupby("dx_sub", observed=True)[["age", "sex", "site"]].describe())
display(metadata_hcp_ep.groupby("dx", observed=True)[["age", "sex", "site"]].describe())

dx
SCZ     184
CTRL     68
Name: count, dtype: int64

dx
SCZ     184
CTRL     67
Name: count, dtype: int64

dx_apd_current
SCZ_noapd    111
SCZ_apd       73
CTRL          67
Name: count, dtype: int64

dx_apd_lifetime
SCZ_apd      133
CTRL          67
SCZ_noapd     51
Name: count, dtype: int64

age                                                          
        count       mean       std    min     25%    50%     75%    max
dx_sub                                                                 
CTRL     67.0  24.272388  4.085239  16.83  20.875  23.92  25.915  35.67
SCZ     127.0  22.707480  3.399370  16.67  20.210  22.08  24.670  34.08
SCA      57.0  23.959474  4.071215  17.58  20.830  22.83  26.330  34.83

age                                                          
      count       mean       std    min     25%    50%     75%    max
dx                                                                   
CTRL   67.0  24.272388  4.085239  16.83  20.875  23.92  25.915  35.67
SCZ   184.0  23.095326  3.655879  16.67  20.330  22.42  24.960  34.83